# Options Scanner v4 — Phase 1 & 2
**CALL and PUT scanner with 12-point quality scoring gate**

Aligned with Jason Brown's PTU trading framework.

A signal is only generated when conviction score >= 6/12.
If nothing qualifies, the scanner tells you to wait.
Every signal includes exact entry, stop, target, and time stop.

---

| Score | Conviction | Action |
|-------|------------|--------|
| 10-12 | VERY HIGH  | Strong candidate |
| 8-9   | HIGH       | Good candidate |
| 6-7   | MODERATE   | Minimum threshold — smaller size |
| < 6   | NONE       | No signal — wait |

**Scoring breakdown (12 pts total):**
Regime (2) + RSI (2) + Trend (2) + Volume (2) + Weekly (1) + Support/Resistance (1) + MACD (1) + Bollinger (1)

**PUT signals in BULLISH market require 9+/12** — broken stock in a strong market rule.

**Execution:**
- Single leg calls/puts → Robinhood or thinkorswim
- Bull call spreads → thinkorswim only
- Stop order type → **Stop Limit** (NOT limit sell)

In [1]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy'])
print('Dependencies ready')

Dependencies ready


In [49]:
# ─────────────────────────────────────────────────────────────
# FORCE RELOAD CELL — run this whenever options_scanner_v4.py
# is updated on GitHub. DO NOT DELETE.
# ─────────────────────────────────────────────────────────────
import sys, os, time

if 'options_scanner_v4' in sys.modules:
    del sys.modules['options_scanner_v4']
if os.path.exists('options_scanner_v4.py'):
    os.remove('options_scanner_v4.py')

url = f"https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py?t={int(time.time())}"
!wget -q -O options_scanner_v4.py "{url}"

# Verify correct version loaded
with open('options_scanner_v4.py') as f:
    content = f.read()

checks = ['target_width', 'min_reward_risk', '2.0', 'also_qualified', 'hard_fail']
for c in checks:
    print(f"{'✓' if c in content else '✗'} {c}")

from options_scanner_v4 import run_scan, print_results, deep_dive, ALL_SYMBOLS, CONFIG, find_best_call
print('\n✓ Scanner reloaded successfully')

✓ target_width
✓ min_reward_risk
✓ 2.0
✓ also_qualified
✓ hard_fail

✓ Scanner reloaded successfully


In [33]:
# Cell 2 — Load scanner from GitHub
# Replace YOUR_USERNAME with kevin-mumaw
!wget -q https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py

from options_scanner_v4 import (
    run_scan, print_results, deep_dive,
    ALL_SYMBOLS, WATCHLIST, CONFIG
)
print('Scanner loaded')
print(f'Watchlist: {len(ALL_SYMBOLS)} symbols')
print('Groups:', list(WATCHLIST.keys()))

Scanner loaded
Watchlist: 34 symbols
Groups: ['tier1_affordable', 'tier2_sweet_spot', 'tier3_mid_price', 'tier4_high_price']


In [21]:
# Cell 3 — Configure
# Edit account size here. Everything else auto-adjusts.

MY_CONFIG = {**CONFIG}  # copy defaults

MY_CONFIG['account_size'] = 5000.00  # update as your account grows

# Optional: scan only specific symbols instead of full watchlist
# MY_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM']
MY_SYMBOLS = None  # None = scan all 25 symbols

print(f'Account: ${MY_CONFIG["account_size"]:,.0f}')
print(f'Min score to generate signal: {MY_CONFIG["min_score"]}/10')
print(f'Stop loss: -{int(MY_CONFIG["stop_loss_pct"]*100)}% | '
      f'Profit target: +{int(MY_CONFIG["profit_target_pct"]*100)}% | '
      f'Time stop: {MY_CONFIG["time_stop_dte"]} DTE')

Account: $5,000
Min score to generate signal: 6/10
Stop loss: -35% | Profit target: +75% | Time stop: 21 DTE


In [34]:
# Cell 4 — Run the scan
# This is the main cell. Run this daily.
results = run_scan(symbols=MY_SYMBOLS, config=MY_CONFIG)
print_results(results)


  Fetching market regime (QQQ)...
  Regime: BULLISH — QQQ 738.31 | MA50 652.9 (+13.1%) | MA200 616.82 (+19.7%)

  Scanning 34 symbols.................................. done.


██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — Phase 1  v4.23  |  2026-05-30 20:46 ET
  Account: $5,000  |  Scanned: 34 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 738.31 | MA50 652.9 (+13.1%) | MA200 616.82 (+19.7%)

  📋 Market extended above MA50 — proceed only with highest conviction setups

  5 signal(s) found:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SIGNAL #1  |  BAC — BUY CALL  |  Score: 9/10  [HIGH]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  WHY THIS TRADE:
    ✓ Regime [2/2] BULLISH — QQQ above both MAs
    ~ RSI     [1/2] Neutral — RSI 52 (room to run)
    ✓ Trend   [2/2] Clean uptrend — price > MA20 > MA50
    ✓ Weekly  [1/1] Weekly uptrend confirmed
    ~ Vo

In [23]:
# Cell 5 — Deep dive on a specific symbol
# Use this to investigate any symbol in detail,
# whether it appeared in the scan or not.

SYMBOL = 'SOFI'  # change this

deep_dive(SYMBOL, config=MY_CONFIG)


  Fetching regime and analyzing SOFI...

██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — Phase 1  v4.22  |  2026-05-31 00:42
  Account: $5,000  |  Scanned: 1 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 738.31 | MA50 652.9 (+13.1%) | MA200 616.82 (+19.7%)

  📋 Market extended above MA50 — proceed only with highest conviction setups

──────────────────────────────────────────────────────────────
  NO QUALIFYING SETUPS TODAY
  Recommendation: Wait. Do not force a trade.

  Best scores today:
    SOFI    5/10  Trend: MIXED       RSI: 69.6

──────────────────────────────────────────────────────────────



In [24]:
# Cell 6 — Review all scores from last scan
# See every symbol's score ranked highest to lowest.

import pandas as pd

rows = []
for r in results['all_results']:
    if not r.get('error'):
        rows.append({
            'Symbol':    r['symbol'],
            'Score':     r['score'],
            'Conviction':r['conviction'],
            'Trend':     r.get('trend','—'),
            'RSI':       r.get('rsi','—'),
            'Volume':    r.get('vol_data',{}).get('label','—'),
            'Weekly':    r.get('weekly',{}).get('trend','—'),
            'Has Option':r.get('option') is not None,
        })

df = pd.DataFrame(rows).sort_values('Score', ascending=False)
print(df.to_string(index=False))

Symbol  Score Conviction     Trend  RSI            Volume    Weekly  Has Option
   BAC      9       HIGH   UPTREND 52.2           NEUTRAL   UPTREND        True
     C      8       HIGH   UPTREND 50.9           NEUTRAL   UPTREND       False
   AAL      7   MODERATE   UPTREND 64.1      ACCUMULATION     MIXED       False
   XLF      7   MODERATE   UPTREND 53.7           NEUTRAL     MIXED        True
    KO      7   MODERATE     MIXED 53.2           NEUTRAL   UPTREND        True
   DIS      7   MODERATE     MIXED 33.2           NEUTRAL     MIXED       False
  UBER      7   MODERATE     MIXED 25.5           NEUTRAL DOWNTREND        True
   AMD      7   MODERATE   UPTREND 67.0           NEUTRAL   UPTREND       False
    MU      7   MODERATE   UPTREND 70.1 MILD_ACCUMULATION   UPTREND       False
  AAPL      7   MODERATE   UPTREND 84.3           NEUTRAL   UPTREND       False
 GOOGL      7   MODERATE     MIXED 35.0           NEUTRAL   UPTREND       False
    GS      7   MODERATE   UPTREND 75.6 

In [ ]:
# Cell 7 — Trade Journal (entry, exit, and re-entry)
import os
if os.path.exists("trade_journal.csv"):
    os.remove("trade_journal.csv")
    print("Journal cleared")

# Trade 1 — Original buy 5/26
trade_id_1 = log_entry(
    symbol         = "BAC",
    direction      = "BUY CALL",
    trade_type     = "SINGLE LEG",
    score          = 9,
    conviction     = "HIGH",
    market_regime  = "BULLISH",
    entry_premium  = 3.45,
    stop_price     = 2.24,
    target_price   = 6.04,
    time_stop_date = "2026-06-26",
    contracts      = 1,
    total_cost     = 345,
    notes          = "First live trade — entered 5/26"
)

log_exit(
    trade_id     = trade_id_1,
    exit_premium = 3.55,
    exit_reason  = "MANUAL",
    notes        = "Accidental exit 5/27 at open — limit sell error. Should have been stop limit."
)

# Trade 2 — Re-entry 5/27
trade_id_2 = log_entry(
    symbol         = "BAC",
    direction      = "BUY CALL",
    trade_type     = "SINGLE LEG",
    score          = 9,
    conviction     = "HIGH",
    market_regime  = "BULLISH",
    entry_premium  = 3.45,
    stop_price     = 2.24,
    target_price   = 6.04,
    time_stop_date = "2026-06-26",
    contracts      = 1,
    total_cost     = 345,
    notes          = "Re-entry 5/27 after accidental exit"
)

print(f"\nOpen trade ID: {trade_id_2}")
show_journal()

Journal cleared

  ✓ Trade logged: BAC-202605272201
    BAC BUY CALL | Entry: $3.45 | Stop: $2.24 | Target: $6.04
    Contracts: 1 | Cost: $345 | Time stop: 2026-06-26

  ✓ Trade closed: BAC-202605272201
    Exit: $3.55 | Reason: MANUAL
    P&L: $+10.00 (+2.9%) — WIN

  ✓ Trade logged: BAC-202605272201
    BAC BUY CALL | Entry: $3.45 | Stop: $2.24 | Target: $6.04
    Contracts: 1 | Cost: $345 | Time stop: 2026-06-26

Open trade ID: BAC-202605272201

──────────────────────────────────────────────────────────────
  TRADE JOURNAL  |  2 trade(s)  |  Filter: ALL
──────────────────────────────────────────────────────────────
  Open: 1  |  Wins: 1  |  Losses: 0  |  Win rate: 100.0%  |  Total P&L: $+10.00
──────────────────────────────────────────────────────────────

  🟢 BAC-202605272201
    BAC BUY CALL | Score: 9/12 | HIGH
    Entry: $3.45 × 1 contracts = $345 | 2026-05-27 18:01 ET
    Exit:  $3.55 | MANUAL | 2026-05-27 18:01 ET  P&L: $+10.00
    Notes: Accidental exit 5/27 at open — limit 

In [ ]:
# Cell 9 — View journal anytime
show_journal()
# show_journal("OPEN")
# show_journal("WIN")


──────────────────────────────────────────────────────────────
  TRADE JOURNAL  |  2 trade(s)  |  Filter: ALL
──────────────────────────────────────────────────────────────
  Open: 1  |  Wins: 1  |  Losses: 0  |  Win rate: 100.0%  |  Total P&L: $+10.00
──────────────────────────────────────────────────────────────

  🔵 BAC-202605272150
    BAC BUY CALL | Score: 9/12 | HIGH
    Entry: $3.45 × 1 contracts = $345 | 2026-05-27 17:50 ET
    Stop: $2.24 | Target: $6.04 | Time stop: 2026-06-26
    Notes: Re-entry after accidental exit at 3.55 on 5/27 (limit sell error). Original entry 5/26 at 3.45, net difference +$9.92 after fees.

  🟢 BAC-202605272154
    BAC BUY CALL | Score: 9/12 | HIGH
    Entry: $3.45 × 1 contracts = $345 | 2026-05-27 17:54 ET
    Exit:  $3.55 | MANUAL | 2026-05-27 17:54 ET  P&L: $+10.00
    Notes: Accidental exit — limit sell triggered at open. Should have been stop limit order.

──────────────────────────────────────────────────────────────



In [51]:
# Cell 10 — Backtest
from options_scanner_v4 import run_backtest, print_backtest_results, export_backtest

df = run_backtest(
    symbols    = ALL_SYMBOLS,
    start_date = "2023-01-01",
    end_date   = "2024-12-31",
    min_score  = 7,   # matches scanner CONFIG
    stop_pct   = 0.35,
)
print_backtest_results(df)


  BACKTEST — v4 Scoring System
  Period: 2023-01-01 to 2024-12-31
  Symbols: 34  |  Min score: 7/12
  Hold: 17d  |  Stop: -35%  |  Target: +75%

  Fetching QQQ regime data...
  BAC... 12 trades
  F... 16 trades
  PLTR... 7 trades
  T... 10 trades
  PFE... 8 trades
  AAL... 13 trades
  SOFI... 5 trades
  XLF... 13 trades
  KO... 13 trades
  DIS... 8 trades
  NKE... 10 trades
  UBER... 12 trades
  AMD... 13 trades
  INTC... 11 trades
  WFC... 13 trades
  C... 13 trades
  MU... 19 trades
  AAPL... 12 trades
  GOOGL... 12 trades
  JPM... 11 trades
  V... 12 trades
  MA... 13 trades
  XOM... 15 trades
  CVX... 16 trades
  UNH... 12 trades
  JNJ... 15 trades
  GS... 13 trades
  MSFT... 13 trades
  AMZN... 12 trades
  META... 11 trades
  NVDA... 10 trades
  COST... 15 trades
  SPY... 11 trades
  QQQ... 13 trades

  BACKTEST RESULTS — v4 Scoring System
  Total trades   : 412
  Win rate       : 49.3%  (203W / 209L)
  Total P&L      : $+40,135.45
  Avg win        : $+385.87
  Avg loss       : $

In [29]:
import requests
url = "https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py"
r = requests.get(url)
print(r.status_code)
print(r.text[:200])


200
"""
options_scanner_v4.py
Phase 1 — CALL-side scanner with quality scoring gate.
Aligned with Jason Brown's PTU trading principles.

Design philosophy:
  - Only generate a signal


In [30]:
import sys, os, importlib

# Nuclear clear
for mod in list(sys.modules.keys()):
    if 'options_scanner' in mod:
        del sys.modules[mod]

if os.path.exists('options_scanner_v4.py'):
    os.remove('options_scanner_v4.py')
    print("Deleted old file")

import requests
url = "https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py"
r = requests.get(url)
with open('options_scanner_v4.py', 'w') as f:
    f.write(r.text)
print(f"Downloaded {len(r.text)} bytes")

# Verify version
import options_scanner_v4 as s
print(f"VERSION: {s.VERSION}")

Deleted old file
Downloaded 112461 bytes
VERSION: 4.23


In [42]:
import options_scanner_v4 as s
print(s.VERSION)


4.24
